# resnet-stem — worked example 3: Stem conv has no bias before BatchNorm

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `resnet-stem`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

The stem's first conv is created with `bias=False`. A bias term is redundant when immediately followed by BatchNorm, because BatchNorm subtracts the batch mean (cancelling any constant shift) and then re-introduces its own learnable shift `beta`. Setting `bias=False` saves parameters without changing the function.

## Worked solution

We build the stem and inspect the first conv's `bias` attribute, which is `None` because we passed `bias=False`. We then count parameters: the conv contributes only its weight `(64*3*7*7 = 9408)` with no bias vector, and the BatchNorm contributes its weight and bias `(64 + 64)`. We print the conv bias status and the BatchNorm affine parameter count to make the redundancy concrete — the shift lives in BatchNorm, not the conv.

In [ ]:
import torch.nn as nn


def build_stem():
    return nn.Sequential(
        nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False),
        nn.BatchNorm2d(64),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
    )


stem = build_stem()
conv = stem[0]
bn = stem[1]
print('conv bias is None:', conv.bias is None)
print('conv weight numel:', conv.weight.numel())
print('bn affine params (weight+bias):', bn.weight.numel() + bn.bias.numel())